In [37]:
import joblib
import numpy as np
import pandas as pd
import os

In [38]:
def load_app(filepath):
    """
    Application locale — ~25-40 Hz.
    Colonnes attendues : time (Unix s), angle1_l/r (°H), angle2_l/r (°V),
                         gaze_x/y_left/right (pixels), fix_x, fix_y.
    Résolution écran : 1366×768.
    Blinks/pertes : valeurs aux limites de l'écran (≥1350 ou ≥750 px).
    """
    df = pd.read_csv(filepath)

    df['t_s'] = df['time'] - df['time'].iloc[0]   # secondes relatives

    if 'angle1_l' in df.columns:
        df['x'] = (df['angle1_l'] + df['angle1_r']) / 2   # degrés horizontaux
        df['y'] = (df['angle2_l'] + df['angle2_r']) / 2   # degrés verticaux
        df['x_left'] = df['angle1_l']
        df['y_left'] = df['angle2_l']
        df['x_right'] = df['angle1_r']
        df['y_right'] = df['angle2_r']

    # df['pupil'] = np.nan

    return df[['t_s', 'x', 'y', 'x_left', 'y_left', 'x_right', 'y_right']].reset_index(drop=True)


In [46]:
model = joblib.load('randomforest.pkl')

df = load_app('../eyes-tracking/exports/gaze_log(9).csv')
df

/Users/maelle/.pyenv/versions/3.10.6/envs/dyslexia/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/maelle/.pyenv/versions/3.10.6/envs/dyslexia/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/maelle/.pyenv/versions/3.10.6/envs/dyslexia/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarnin

,t_s,x,y,x_left,y_left,x_right,y_right
0,0.000000,6.187569,-3.758283,3.203829,-5.492650,9.171310,-2.023916
1,0.061563,5.606221,-2.368739,2.094849,-3.782225,9.117594,-0.955252
2,0.154790,4.853178,-4.337964,0.904869,-5.824714,8.801487,-2.851213
3,0.306824,4.925862,-2.789745,1.192499,-5.012390,8.659225,-0.567100
4,0.338877,3.798027,-2.330280,0.262704,-4.201918,7.333350,-0.458641
...,...,...,...,...,...,...,...
624,57.230822,3.111524,-8.648809,1.546627,-11.557188,4.676421,-5.740429
625,57.263112,-0.158244,-8.106663,-3.597009,-9.629653,3.280521,-6.583674
626,57.389514,0.434976,-8.013837,-3.760526,-9.634260,4.630478,-6.393414
627,57.446626,2.032432,-7.562914,-2.138342,-8.396266,6.203206,-6.729562


In [40]:
def extract_features(df, subject_id=None, label=None, source=None):
    """
    Calcule les 9 features comportementales à partir d'un DataFrame normalisé
    (colonnes : t_s, x, y).

    Toutes les features sont des ratios ou coefficients sans unité,
    directement comparables entre DS1, DS2 et l'application.

    Paramètres
    ----------
    df         : DataFrame retourné par load_dataset1/2/app
    subject_id : identifiant du sujet (optionnel)
    label      : 1 = dyslexique, 0 = contrôle, None = inconnu
    source     : 'ds1', 'ds2' ou 'app'

    Retourne
    --------
    dict avec les features + métadonnées (subject_id, dyslexia, source,
    duration_s, n_samples) ou None si données insuffisantes.
    """
    x = df['x'].values
    y = df['y'].values
    t = df['t_s'].values

    if len(x) < 10:
        return None

    # ── Vélocité instantanée ──────────────────────────────────
    dx = np.diff(x)
    dy = np.diff(y)
    dt = np.diff(t)
    dt = np.where(dt < 1e-6, 1e-6, dt)
    raw_vel = np.sqrt(dx**2 + dy**2) / dt

    # Clipper les artefacts (>99.5e percentile = transitions inter-textes, clignements résiduels)
    vel_ceiling = np.percentile(raw_vel, 99.5)
    velocity = np.clip(raw_vel, 0, vel_ceiling)

    # ── Saccades vs fixations ─────────────────────────────────
    # Seuil adaptatif : 75e percentile de la session (robuste aux différences d'unité)
    v_thresh    = np.percentile(velocity, 75)
    is_saccade  = velocity > v_thresh
    is_fixation = ~is_saccade

    # ── Directions ───────────────────────────────────────────
    regression_mask = (dx < 0) & is_saccade   # vers la gauche
    forward_mask    = (dx > 0) & is_saccade   # vers la droite

    n_saccades   = int(np.sum(is_saccade))
    n_regression = int(np.sum(regression_mask))
    n_forward    = int(np.sum(forward_mask))

    fwd_amp = np.mean(np.abs(dx[forward_mask]))    if n_forward    > 0 else 1e-9
    reg_amp = np.mean(np.abs(dx[regression_mask])) if n_regression > 0 else 0.0

    vel_mean   = float(np.mean(velocity))
    vel_std    = float(np.std(velocity))

    # ── Irrégularité des saccades ─────────────────────────────
    saccade_idx = np.where(is_saccade)[0]
    if len(saccade_idx) > 2:
        ipi = np.diff(saccade_idx)   # inter-pulse intervals
        saccade_reg = float(np.std(ipi) / max(np.mean(ipi), 1))
    else:
        saccade_reg = 0.0

    # ── Dérive verticale ──────────────────────────────────────
    mid = len(y) // 2
    y_range = float(np.ptp(y))
    y_drift = float((np.mean(y[mid:]) - np.mean(y[:mid])) / (y_range + 1e-9))

    # ── Assemblage ────────────────────────────────────────────
    feat = {
        'saccade_rate':       float(np.mean(is_saccade)),
        'fixation_prop':      float(np.mean(is_fixation)),
        'regression_rate':    float(n_regression / max(n_saccades, 1)),
        'reg_fwd_ratio':      float(reg_amp / fwd_amp),
        'vel_cv':             float(vel_std / vel_mean if vel_mean > 0 else 0),
        'x_spread_ratio':     float(np.std(x) / (np.ptp(x) + 1e-9)),
        'y_spread_ratio':     float(np.std(y) / (y_range + 1e-9)),
        'y_drift':            y_drift,
        'saccade_regularity': saccade_reg,
        # Métadonnées
        'duration_s':  float(t[-1] - t[0]),
        'n_samples':   len(x),
    }

    if subject_id is not None: feat['subject_id'] = subject_id
    if label      is not None: feat['dyslexia']   = label
    if source     is not None: feat['source']      = source

    return feat

In [47]:
feat = extract_features(df)
feat

{'saccade_rate': 0.25,
 'fixation_prop': 0.75,
 'regression_rate': 0.5286624203821656,
 'reg_fwd_ratio': 1.4081853253564183,
 'vel_cv': 0.9235877905033831,
 'x_spread_ratio': 0.16711717883476654,
 'y_spread_ratio': 0.14678836011248927,
 'y_drift': -0.09754131066471729,
 'saccade_regularity': 0.7073254339210345,
 'duration_s': 57.54734969139099,
 'n_samples': 629}

In [48]:
feat_df = pd.DataFrame([feat])
feat_df

,saccade_rate,fixation_prop,regression_rate,reg_fwd_ratio,vel_cv,x_spread_ratio,y_spread_ratio,y_drift,saccade_regularity,duration_s,n_samples
0,0.25,0.75,0.528662,1.408185,0.923588,0.167117,0.146788,-0.097541,0.707325,57.54735,629


In [49]:
FEATURES = ['regression_rate', 'reg_fwd_ratio', 'vel_cv',
            'x_spread_ratio', 'y_spread_ratio', 'y_drift', 'saccade_regularity']

In [50]:
x = np.array([[feat[f] for f in FEATURES]])
x

array([[ 0.52866242,  1.40818533,  0.92358779,  0.16711718,  0.14678836,
        -0.09754131,  0.70732543]])

In [51]:
prob = model.predict_proba(x)[0]
pred = int(model.predict(x)[0])
prob, pred

(array([0.58095167, 0.41904833]), 0)